<span style="color:lightgreen">

# Notebook extra 1

En este notebook se experimenta con los parámetros de transcripción con traducción de whisper.

Covost2 pt-en
</span>

# ST Baseline experiment using Whisper and Europarl-ST (Spanish-English)


In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for speech translation on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST) (using Spanish-English speech data).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing evaluation figures: WER and BLEU)

In [7]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import whisper
from whisper.normalizers import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

model = whisper.load_model("base")

Load Europarl-ST Spanish-English test audio dataset

In [9]:
from datasets import load_dataset

raw_datasets = load_dataset("facebook/covost2", CONFIG.trans_code, data_dir=CONFIG.corpus_path)
data = raw_datasets["test"]

print(data)

Dataset({
    features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
    num_rows: 4023
})


<p style="page-break-after:always;"></p>

<span style="color:lightgreen">

Ahora definimos parámetros para el experimento

</span>

In [10]:

experiments = [
    {
        "name": "Experiment 1: Default parameters",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 2: Low temperature (more deterministic)",
        "normalize": True,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 3: Lenient thresholds",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 3.0,
            "logprob_threshold": -1.5,
            "no_speech_threshold": 0.8,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 4: No conditioning on previous text",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": False,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 5: With initial prompt and strict thresholds",
        "normalize": True,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.0,
            "logprob_threshold": -0.5,
            "no_speech_threshold": 0.5,
            "condition_on_previous_text": True,
            "initial_prompt": "This is a European Parliament speech about politics and legislation."
        }
    },

    # WITH NO NORMALIZATION
    {
        "name": "Experiment 1: Default parameters",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 2: Low temperature (more deterministic)",
        "normalize": False,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 3: Lenient thresholds",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 3.0,
            "logprob_threshold": -1.5,
            "no_speech_threshold": 0.8,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 4: No conditioning on previous text",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": False,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 5: With initial prompt and strict thresholds",
        "normalize": False,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.0,
            "logprob_threshold": -0.5,
            "no_speech_threshold": 0.5,
            "condition_on_previous_text": True,
            "initial_prompt": "This is a European Parliament speech about politics and legislation."
        }
    }
]

# Display experiments
for exp in experiments:
    print(f"\n{exp['name']}")
    for key, value in exp['params'].items():
        print(f"  {key}: {value}")


Experiment 1: Default parameters
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: True
  initial_prompt: None

Experiment 2: Low temperature (more deterministic)
  temperature: 0.0
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: True
  initial_prompt: None

Experiment 3: Lenient thresholds
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 3.0
  logprob_threshold: -1.5
  no_speech_threshold: 0.8
  condition_on_previous_text: True
  initial_prompt: None

Experiment 4: No conditioning on previous text
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: False
  initial_prompt: None

Experiment 5: With initial prompt and strict thresholds
  temperature: 0.0
  compression_

<span style="color:lightgreen">
mute warnings

<span>

In [11]:
import torch
import warnings
import os
import logging

# Set Tensor Core precision for RTX 5070
torch.set_float32_matmul_precision('high')

# Suppress PyTorch Lightning tips and info messages
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

# Alternative: suppress all Lightning logs
# os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'

In [ ]:
from maikol_utils.print_utils import print_separator, print_color
from evaluate import load

normalizer = BasicTextNormalizer()
bleu_metric = load("sacrebleu")
comet_metric = load('comet')
bleu_scores, comet_scores = {}, {}
experiments_results = {}

for exp in experiments:
    print_separator(exp["name"])
    
    experiments_results[exp["name"]] = {}
    translations = []

    for sample in tqdm(data["file"]):   
        translations.append((model.transcribe(sample, language=CONFIG.src_name, task="translate"))['text'])

    data_df = pd.DataFrame(dict(translation=translations, reference=data["translation"], source=data["sentence"]))

    if exp["normalize"]:
        data_df["translation_ready"] = [normalizer(text) for text in data_df["translation"]]
        data_df["reference_ready"] = [normalizer(text) for text in data_df["reference"]]
        data_df["source_ready"] = [normalizer(text) for text in data_df["source"]]
    else:
        data_df["translation_ready"] = data_df["translation"]
        data_df["reference_ready"] = data_df["reference"]
        data_df["source_ready"] = data_df["source"]

    bleu_result = bleu_metric.compute(predictions=data_df["translation_ready"], references=data_df["reference_ready"])
    comet_score = comet_metric.compute(predictions=data_df["translation_ready"], references=data_df["reference_ready"], sources=data_df["source_ready"])

    bleu_scores[exp["name"]] = bleu_result["score"]
    comet_scores[exp["name"]] = comet_score['mean_score']

    print_color(f'BLEU score: {bleu_result["score"]:.1f}', color="cyan")
    print_color(f"COMET: {comet_score['mean_score']:.2%}", color="cyan")

    
    experiments_results[exp["name"]]["translations"] = translations
    experiments_results[exp["name"]]["references"] = data_df["reference"]
    experiments_results[exp["name"]]["sources"] = data_df["source"]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Encoder model frozen.


________________________________________________________________
                Experiment 1: Default parameters                



  0%|          | 0/4023 [00:00<?, ?it/s]

BLEU score: 22.9
COMET: 62.79%
________________________________________________________________
       Experiment 2: Low temperature (more deterministic)       



  0%|          | 0/4023 [00:00<?, ?it/s]

BLEU score: 22.9
COMET: 62.64%
________________________________________________________________
                Experiment 3: Lenient thresholds                



  0%|          | 0/4023 [00:00<?, ?it/s]

<span style="color:lightgreen">

### Best scores

<span>

In [ ]:
# Best bleu

best_bleu_exp = max(bleu_scores, key=bleu_scores.get)
best_bleu_score = bleu_scores[best_bleu_exp]

print_separator("Best Experiment by BLEU Score")
print_color(f'Best Experiment: {best_bleu_exp} with BLEU score: {best_bleu_score:.1f}', color="green")

# Best comet
best_comet_exp = max(comet_scores, key=comet_scores.get)
best_comet_score = comet_scores[best_comet_exp]

print_separator("Best Experiment by COMET Score")
print_color(f'Best Experiment: {best_comet_exp} with COMET score: {best_comet_score:.2%}', color="green")

________________________________________________________________
                 Best Experiment by BLEU Score                  

Best Experiment: Experiment 3: Lenient thresholds with BLEU score: 20.1
________________________________________________________________
                 Best Experiment by COMET Score                 

Best Experiment: Experiment 3: Lenient thresholds with COMET score: 61.15%


'\x1bBest Experiment: Experiment 3: Lenient thresholds with COMET score: 61.15%\x1b'

<p style="page-break-after:always;"></p>

### Best results bleu

In [ ]:
data = pd.DataFrame(dict(translation=experiments_results[best_bleu_exp]["translations"], reference=experiments_results[best_bleu_exp]["references"], source=experiments_results[best_bleu_exp]["sources"]))
pd.set_option('display.max_colwidth', None)
data.head(2)

,translation,reference,source
0,"The National Security Council, the officials and even the Spanish police, are being mistreated by the European Federation of Fouold, but in the case of the particular one. Those initiatives aggravate the sanctions that occur to the ordinary justice. This medieval conception, this law of the law of the law, is incompatible with the law, with the European institutions, from the ones we react and we will end up doing that, because those medieval gentlemen, arbitraries of Orca and Cuchillo, have to put the law in line with the respect to the law and the great ordinary processes of our European Union. Just a few times.","Atlético Madrid, its fans and even the Spanish police are being mistreated by the Union of European Football Associations. However, the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts.\nThis mediaeval concept of one law for me and another for you is contrary to our law and the European institutions. We must therefore react. In fact, we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our Europe.","El Atlético de Madrid , los aficionados e incluso la policía española están siendo maltratados por la Federación Europea de Fútbol . Pero el caso transciende lo particular , pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria .\nEsta concepción medieval , esta ley del embudo es incompatible con el Derecho y con las instituciones europeas , desde las que hemos de reaccionar . Lo acabaremos haciendo , pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al Derecho y las garantías procesales ordinarias de nuestra Europa .\n"
1,"Thank you, Mr. President, Mr. Commissioner. Terrorism is a huge global phenomenon and the act of serious danger is also organized. Therefore, all the media have to be proportional and have to fight for their effectiveness. I have taken a good note of the answers that were left to the questions that were the next. It is true that there are guaranteees, it is true that it is a delicate issue, but it is not true that it is absolutely inexcusable to form a globalized and harmonized response. For some, terrorism spills a little away, it worries more the individual's guarantees, it worries the individuals and the colleagues. And that is absolutely necessary that we start where we can. If we start through the transport area where the companies and action of those data start there. If we look at the ancient, we see what is the application habit, we start through the relations of international sports. And we have to follow the previous ones because terrorists, many times, they do not come from outside and they do not come from inside. They ask them in the United States and they ask them in addition that this is how we have to plan.","Mr President, Commissioner, terrorism and serious organised crime are global phenomena. The means for fighting these must therefore be proportional and effective.\nI took due note of the answers given to the questions. These answers were quite correct: it is true that guarantees must be demanded and that this is a delicate issue. However, it is also true that it is absolutely inexcusable to provide a globalised and harmonised response.\nThose people who are somewhat detached from terrorism are more concerned about individual guarantees. My concern is for both individual and collective guarantees. It is absolutely vital that we start where we can. If we have to start with air transport, given that air carriers already have this data, then that is where we must start.\nWe will demand guarantees, we will assess the scope and we will start with international transport. However, it should be noted that we will then move on to domestic transport because terrorists very often do not come from outside, but are home-

Automatic translations, references and sources are normalized using the Whisper basic text standardisation/normalization module

In [ ]:
normalizer = BasicTextNormalizer()

data["translation_clean"] = [normalizer(text) for text in data["translation"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data["source_clean"] = [normalizer(text) for text in data["source"]]
data.head(2)

,translation,reference,source,translation_clean,reference_clean,source_clean
0,"The National Security Council, the officials and even the Spanish police, are being mistreated by the European Federation of Fouold, but in the case of the particular one. Those initiatives aggravate the sanctions that occur to the ordinary justice. This medieval conception, this law of the law of the law, is incompatible with the law, with the European institutions, from the ones we react and we will end up doing that, because those medieval gentlemen, arbitraries of Orca and Cuchillo, have to put the law in line with the respect to the law and the great ordinary processes of our European Union. Just a few times.","Atlético Madrid, its fans and even the Spanish police are being mistreated by the Union of European Football Associations. However, the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts.\nThis mediaeval concept of one law for me and another for you is contrary to our law and the European institutions. We must therefore react. In fact, we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our Europe.","El Atlético de Madrid , los aficionados e incluso la policía española están siendo maltratados por la Federación Europea de Fútbol . Pero el caso transciende lo particular , pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria .\nEsta concepción medieval , esta ley del embudo es incompatible con el Derecho y con las instituciones europeas , desde las que hemos de reaccionar . Lo acabaremos haciendo , pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al Derecho y las garantías procesales ordinarias de nuestra Europa .\n",the national security council the officials and even the spanish police are being mistreated by the european federation of fouold but in the case of the particular one those initiatives aggravate the sanctions that occur to the ordinary justice this medieval conception this law of the law of the law is incompatible with the law with the european institutions from the ones we react and we will end up doing that because those medieval gentlemen arbitraries of orca and cuchillo have to put the law in line with the respect to the law and the great ordinary processes of our european union just a few times,atlético madrid its fans and even the spanish police are being mistreated by the union of european football associations however the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts this mediaeval concept of one law for me and another for you is contrary to our law and the european institutions we must therefore react in fact we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our europe,el atlético de madrid los aficionados e incluso la policía española están siendo maltratados por la federación europea de fútbol pero el caso transciende lo particular pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria esta concepción medieval esta ley del embudo es incompatible con el derecho y con las instituciones europeas desde las que hemos de reaccionar lo acabaremos haciendo pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al derecho y las garantías procesales ordinarias de nuestra europa
1,"Thank you, Mr. President, Mr. Commissioner. Terrorism is a huge global phenomenon and the act of serious danger is also organized. Therefore, all the media have to be proportional and have to fight for their effectiveness. I have taken a good note of the answers that were left to the questions that were the next. It is true that there are guaranteees, it is true that 

### Best comet

In [ ]:
data = pd.DataFrame(dict(translation=experiments_results[best_comet_exp]["translations"], reference=experiments_results[best_comet_exp]["references"], source=experiments_results[best_comet_exp]["sources"]))
pd.set_option('display.max_colwidth', None)
data.head(2)

,translation,reference,source
0,"The National Security Council, the officials and even the Spanish police, are being mistreated by the European Federation of Fouold, but in the case of the particular one. Those initiatives aggravate the sanctions that occur to the ordinary justice. This medieval conception, this law of the law of the law, is incompatible with the law, with the European institutions, from the ones we react and we will end up doing that, because those medieval gentlemen, arbitraries of Orca and Cuchillo, have to put the law in line with the respect to the law and the great ordinary processes of our European Union. Just a few times.","Atlético Madrid, its fans and even the Spanish police are being mistreated by the Union of European Football Associations. However, the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts.\nThis mediaeval concept of one law for me and another for you is contrary to our law and the European institutions. We must therefore react. In fact, we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our Europe.","El Atlético de Madrid , los aficionados e incluso la policía española están siendo maltratados por la Federación Europea de Fútbol . Pero el caso transciende lo particular , pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria .\nEsta concepción medieval , esta ley del embudo es incompatible con el Derecho y con las instituciones europeas , desde las que hemos de reaccionar . Lo acabaremos haciendo , pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al Derecho y las garantías procesales ordinarias de nuestra Europa .\n"
1,"Thank you, Mr. President, Mr. Commissioner. Terrorism is a huge global phenomenon and the act of serious danger is also organized. Therefore, all the media have to be proportional and have to fight for their effectiveness. I have taken a good note of the answers that were left to the questions that were the next. It is true that there are guaranteees, it is true that it is a delicate issue, but it is not true that it is absolutely inexcusable to form a globalized and harmonized response. For some, terrorism spills a little away, it worries more the individual's guarantees, it worries the individuals and the colleagues. And that is absolutely necessary that we start where we can. If we start through the transport area where the companies and action of those data start there. If we look at the ancient, we see what is the application habit, we start through the relations of international sports. And we have to follow the previous ones because terrorists, many times, they do not come from outside and they do not come from inside. They ask them in the United States and they ask them in addition that this is how we have to plan.","Mr President, Commissioner, terrorism and serious organised crime are global phenomena. The means for fighting these must therefore be proportional and effective.\nI took due note of the answers given to the questions. These answers were quite correct: it is true that guarantees must be demanded and that this is a delicate issue. However, it is also true that it is absolutely inexcusable to provide a globalised and harmonised response.\nThose people who are somewhat detached from terrorism are more concerned about individual guarantees. My concern is for both individual and collective guarantees. It is absolutely vital that we start where we can. If we have to start with air transport, given that air carriers already have this data, then that is where we must start.\nWe will demand guarantees, we will assess the scope and we will start with international transport. However, it should be noted that we will then move on to domestic transport because terrorists very often do not come from outside, but are home-

In [ ]:
data["translation_clean"] = [normalizer(text) for text in data["translation"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data["source_clean"] = [normalizer(text) for text in data["source"]]
data.head(2)

,translation,reference,source,translation_clean,reference_clean,source_clean
0,"The National Security Council, the officials and even the Spanish police, are being mistreated by the European Federation of Fouold, but in the case of the particular one. Those initiatives aggravate the sanctions that occur to the ordinary justice. This medieval conception, this law of the law of the law, is incompatible with the law, with the European institutions, from the ones we react and we will end up doing that, because those medieval gentlemen, arbitraries of Orca and Cuchillo, have to put the law in line with the respect to the law and the great ordinary processes of our European Union. Just a few times.","Atlético Madrid, its fans and even the Spanish police are being mistreated by the Union of European Football Associations. However, the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts.\nThis mediaeval concept of one law for me and another for you is contrary to our law and the European institutions. We must therefore react. In fact, we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our Europe.","El Atlético de Madrid , los aficionados e incluso la policía española están siendo maltratados por la Federación Europea de Fútbol . Pero el caso transciende lo particular , pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria .\nEsta concepción medieval , esta ley del embudo es incompatible con el Derecho y con las instituciones europeas , desde las que hemos de reaccionar . Lo acabaremos haciendo , pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al Derecho y las garantías procesales ordinarias de nuestra Europa .\n",the national security council the officials and even the spanish police are being mistreated by the european federation of fouold but in the case of the particular one those initiatives aggravate the sanctions that occur to the ordinary justice this medieval conception this law of the law of the law is incompatible with the law with the european institutions from the ones we react and we will end up doing that because those medieval gentlemen arbitraries of orca and cuchillo have to put the law in line with the respect to the law and the great ordinary processes of our european union just a few times,atlético madrid its fans and even the spanish police are being mistreated by the union of european football associations however the problem is wider than this as these federative bodies tend to increase sanctions when people resort to the ordinary courts this mediaeval concept of one law for me and another for you is contrary to our law and the european institutions we must therefore react in fact we will end up having to react as these arbitrary mediaeval tyrants must abide by the law and the ordinary procedural guarantees of our europe,el atlético de madrid los aficionados e incluso la policía española están siendo maltratados por la federación europea de fútbol pero el caso transciende lo particular pues esos órganos federativos agravan las sanciones a quienes recurren a la justicia ordinaria esta concepción medieval esta ley del embudo es incompatible con el derecho y con las instituciones europeas desde las que hemos de reaccionar lo acabaremos haciendo pues esos señores medievales arbitrarios de horca y cuchillo han de ponerse en línea con el respeto al derecho y las garantías procesales ordinarias de nuestra europa
1,"Thank you, Mr. President, Mr. Commissioner. Terrorism is a huge global phenomenon and the act of serious danger is also organized. Therefore, all the media have to be proportional and have to fight for their effectiveness. I have taken a good note of the answers that were left to the questions that were the next. It is true that there are guaranteees, it is true that 

<p style="page-break-after:always;"></p>